/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import netket as nk
import netket.experimental as nkx
import numpy as np
import matplotlib.pyplot as plt
from pyscf import gto, scf, fci
import jax
import jax.numpy as jnp
from flax import nnx
#==============================================================================
# 1. 分子定义 + FCI 精确基准（你的代码）
#==============================================================================
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]

mol = gto.M(atom=geometry, basis='STO-3G')
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot

# FCI 精确解（4个态）
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0])*27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# 转为 NetKet 哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

#==============================================================================
# 2. Hilbert 空间 + 采样器（你的代码）
#==============================================================================
n_orbitals = 2
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=n_orbitals,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)

g = nk.graph.Graph(edges=[(0,1), (2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=4, spin_symmetric=True, sweep_size=64
)

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


In [3]:
#==============================================================================
# 3. 神经网络模型（你的代码完全不变）
#==============================================================================
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

class NESTotalAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals, n_states=2, hidden_dim=8, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_states = n_states
        self.n_spin_orbitals = n_spin_orbitals
        self.single_ansatz_list = [
            SingleStateAnsatz(n_spin_orbitals, hidden_dim, rngs=nnx.Rngs(42+i))
            for i in range(n_states)
        ]

    def __call__(self, x_batch, return_log_psi=False):  # ✅ 加开关
        # 你的原有逻辑
        K = self.n_states
        M = []
        for i in range(K):
            row = [self.single_ansatz_list[j](x_batch[i]) for j in range(K)]
            M.append(jnp.array(row))
        M = jnp.stack(M)
        psi_total = jnp.linalg.det(M)
        log_psi_total = jnp.log(psi_total)  # NetKet 必须用 log

        # ✅ 关键：Sampler 只需要 log_psi（一个值）
        if return_log_psi:
            return log_psi_total  
        # ✅ 算能量时返回两个值
        else:
            return psi_total, M  



In [4]:
#==============================================================================
# 4. 核心：HΨ + 局部能量（你的代码 100% 不变）
#==============================================================================
def Ham_psi(ha: nk.operator.DiscreteOperator, model, x: jnp.array):
    x_primes, mels = ha.get_conn(x)
    psi_vals = jax.vmap(model)(x_primes)
    return jnp.sum(mels * psi_vals)

def Ham_Psi(ha, total_ansatz: NESTotalAnsatz, x_batch):
    K = total_ansatz.n_states
    H_mat = []
    for i in range(K):
        row = []
        for j in range(K):
            v = Ham_psi(ha, total_ansatz.single_ansatz_list[j], x_batch[i])
            row.append(v)
        H_mat.append(row)
    return jnp.array(H_mat, dtype=complex)

def compute_local_energy(ha, total_ansatz: NESTotalAnsatz, x_batch):
    psi, M = total_ansatz(x_batch)
    Hp = Ham_Psi(ha, total_ansatz, x_batch)
    el_mat = jnp.linalg.solve(M, Hp)
    return jnp.trace(el_mat), el_mat

#==============================================================================
# 5. ✅ 论文标准：损失函数 + 自动梯度（JAX 正确求导）
#==============================================================================
@jax.jit
def loss_fn(model_state, samples):
    # 绑定参数
    nnx.update(total_ansatz, model_state)
    
    total_energy = 0.0 + 0j
    n_samples = samples.shape[0]
    
    for xb in samples:
        tr_EL, _ = compute_local_energy(ha, total_ansatz, xb)
        total_energy += tr_EL
    
    avg_energy = total_energy.real / n_samples
    return avg_energy

@jax.jit
def train_step(model_state, opt_state, samples):
    # 🔥 自动计算论文所需的参数梯度
    loss, grads = jax.value_and_grad(loss_fn)(model_state, samples)
    
    # 参数更新（修复：nnx.apply_updates → opt.update）
    # 完全兼容你的 Flax NNX 版本
    model_state = jax.tree_util.tree_map(lambda p, g: p - 0.01 * g, model_state, grads)
    
    return model_state, opt_state, loss


In [9]:
#==============================================================================
# 6. 模型 + 优化器初始化
#==============================================================================
K = 2                  # 同时算基态 + 第一激发态
n_spin_orb = 4
hidden_dim = 8
lr = 0.01
train_steps = 400
sample_batch = 16

# 初始化模型
rngs = nnx.Rngs(42)
total_ansatz = NESTotalAnsatz(n_spin_orb, K, hidden_dim, rngs=rngs)
single_ansatz = SingleStateAnsatz(n_spin_orbitals=4,hidden_dim=4*3,rngs=rngs)
model_state = nnx.state(total_ansatz)

optimizer = nk.optimizer.Sgd(learning_rate=0.05)
opt_state = optimizer.init(model_state)

In [6]:
def forward(state,x):
  y, ffnn_state = nnx.call(state)(x,return_log_psi=True)
  return y


In [11]:
g = nk.graph.Graph(edges=[(0,1), (2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=10, spin_symmetric=True, sweep_size=64
)

def forward(state,x):
  y, ffnn_state = nnx.call(state)(x)
  return y
parameters = nnx.split(single_ansatz)
sampler_state = sampler.init_state(forward, parameters, seed=1)
sampler_state = sampler.reset(forward, parameters, sampler_state)

samples, sampler_state = sampler.sample(
    forward, parameters, state=sampler_state, chain_length=100
)

In [ ]:
# 训练循环
for step in range(train_steps):
    # 1. 采样（完全使用你的代码格式）
    samples1, sampler_state1 = sampler.sample(
        forward, parameters1, state=sampler_state1, chain_length=100
    )
    samples2, sampler_state2 = sampler.sample(
        forward, parameters2, state=sampler_state2, chain_length=100
    )

    # 2. 拼接成 NES-VMC 需要的 x_batch: [K, n_spin_orb] = [2,4]
    # 取每个链的最后一个样本，构造 K=2 组态
    x_batch = jnp.stack([samples1[0, -1], samples2[0, -1]], axis=0)  # shape [2,4]

    # 3. 训练一步（传入构造好的 x_batch）
    model_state, opt_state, loss = train_step(model_state, opt_state, x_batch[None, ...])
    energy_history.append(float(loss))

    # 4. 同步更新 params1 / params2
    nnx.update(total_ansatz, model_state)
    params1, _ = nnx.split(total_ansatz.single_ansatz_list[0])
    params2, _ = nnx.split(total_ansatz.single_ansatz_list[1])

    # 保存能量矩阵
    if step % 10 == 0:
        _, el_mat = compute_local_energy(ha, total_ansatz, x_batch)
        el_mat_list.append(el_mat)
    
    # 打印日志
    if step % 50 == 0:
        print(f"Step {step:3d} | Energy = {loss:.8f} Ha")

In [ ]:
params1, _ = nnx.split(total_ansatz.single_ansatz_list[0])